# L-curve plots for Az-consistent-forward scalar-amplitude inversion

Counterpart to `plt_syn_slip_az_inv_hetmu_nicoya_locking_lc.ipynb` for the cf scheme.

* **Synth L-curves** (cells below): both CK (`nicoyaCKden_une_sm`) and non-CK (`nicoyaden_sm`) plate models. L-curve files are written by the cf forward+inversion scripts (`synth_*_az_consistfwd_*.py`) following the naming `Lcurvesynscaamp_cf_azbd_rs<rho_s>_<meshname>_<slip_pattern>_<pattern_option>_<mu_for>_<mu_inv>.txt`.
* **Real-data L-curves** (later cells): real-GPS interseismic + coseismic inversions, kept unchanged — they use their own (non-cf) inversion scheme and naming.

Switch which L-curve file family to load by changing the `lc_prefix` variable
inside the synth setup cell:
* `Lcurvesynlockbd_azbd_rs`    — original (non-consistent) Az forward
* `Lcurvesynscaamp_azbd_rs`    — scalar-amp Az (no cf)
* `Lcurvesynscaamp_cf_azbd_rs` — scalar-amp Az **consistent forward** (current default)


In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc
from io import StringIO
import utils_plot as utp

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "syn_slip/"

# Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
obs_disp_name = "data/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
                          'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])

In [ ]:
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1
# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email

In [ ]:
# meshnameCK = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshnameCK = "nicoyaCK4"   # same as CK2, but connecting the trench now

# Meshes with even top boundary at 0 depth
# meshnameCK = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone
# meshnameCK = "nicoyaCKden_all"   # based on nicoyaCK3 or 4, but denser mesh size, and all subduction interface

# Mesh with uneven top boundary, left at mean trench depth ~7 km, right at 0 km
meshnameCK = "nicoyaCKden_une_sm"   # uneven top boundary, smaller fault zone, counterpart to 'nicoyaCKden_sm'
# meshnameCK = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface, counterpart to 'nicoyaCKden_all'

# Flag to indicate if using uneven mesh (will be set automatically based on meshname)
use_uneven_mesh = "une" in meshnameCK

### define the pattern of the slip distribution
# slip_pattern = "checker"
slip_pattern = "stripe"

### stripe pattern options
pattern_option = 1  # 1: 2-stripe, shallow-deep; 2: 1-stripe, intermediate
# pattern_option = 2

def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# string for the homogeneous solution
homo_str = f"_mul{round(mu_expression(mu_b))}u{round(mu_expression(mu_b))}"
# string for the contrast between slab and wedge
sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
# string for the original 3d model
het3d_str_ori = f"_DeShon3D_ref1D_{round(1.0)}_hull"
# string for the 3d model, value multiplied by 4, relative a reference
# contrast_factor = 4.0  # amplification factor, too extreme, needs clipping, and not adopted since 03/05/2026
contrast_factor = 2.5  # amplification factor, more reasonable, and adopted since 03/05/2026
# het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}"
# het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}_hull"
het3d_str = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"
# het3d_str_4 = f"_DeShon3D_ref1D_{round(4.0)}_hull"

CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
if CG_mu_deg == 2:
    het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"
    het3d_str = het3d_str + f"_CG{CG_mu_deg}"

rho_s = 1e9

# Filename prefix -- change here to switch between forward strategies:
# "Lcurvesynlockbd_azbd_rs"    -- original (non-consistent) Az forward
# "Lcurvesynscaamp_azbd_rs"    -- scalar amp Az forward (no CF)
# "Lcurvesynscaamp_cf_azbd_rs" -- scalar amp Az consistent-forward
lc_prefix = "Lcurvesynscaamp_cf_azbd_rs"

def _load_lc(fwd_str, inv_str):
    # Load an L-curve file, sort rows by gamma ascending
    fname = f"{lc_prefix}{rho_s:.0e}_{meshnameCK}_{slip_pattern}_{pattern_option}_{fwd_str}_{inv_str}.txt"
    print(fname)
    df = pd.read_csv(datadir + resultpath + fname, sep=r'\s+',
                     names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    return df.sort_values('gamma').reset_index(drop=True)

misfitsCK_hom   = _load_lc(homo_str,  homo_str)
misfitsCK_swsw  = _load_lc(sw_str,    sw_str)
misfitsCK_swhom = _load_lc(sw_str,    homo_str)
misfitsCK_3d3d  = _load_lc(het3d_str, het3d_str)
misfitsCK_3dhom = _load_lc(het3d_str, homo_str)

# ── Preferred γ machinery (file-driven, no hardcoded γ list) ──────────────
# Mirrors the pattern used in plt_slip_inv_hetmu_nicoya_coseis_lc.ipynb:
# pick the row whose 'gamma' column is closest to a requested value, so the
# preferred γ doesn't have to exactly equal one of the γ values that were run.
# Robust to .0e/.1e rounding quirks and small changes to the γ list.
def _idx_for_gamma_syn(df, gamma_pref):
    return int((df['gamma'] - gamma_pref).abs().idxmin())

# Preferred γ per (forward, inversion) μ pairing.  Edit these values to pick
# the L-curve corner per panel; idxCK_* below will resolve to the closest row.
gamma_CK_hom_prefer    = 1.5e2
gamma_CK_swsw_prefer   = 1.5e2
gamma_CK_swhom_prefer  = 1.5e2
gamma_CK_3d3d_prefer   = 1.5e2
gamma_CK_3dhom_prefer  = 1.5e2

idxCK_hom    = _idx_for_gamma_syn(misfitsCK_hom,   gamma_CK_hom_prefer)
idxCK_swsw   = _idx_for_gamma_syn(misfitsCK_swsw,  gamma_CK_swsw_prefer)
idxCK_swhom  = _idx_for_gamma_syn(misfitsCK_swhom, gamma_CK_swhom_prefer)
idxCK_3d3d   = _idx_for_gamma_syn(misfitsCK_3d3d,  gamma_CK_3d3d_prefer)
idxCK_3dhom  = _idx_for_gamma_syn(misfitsCK_3dhom, gamma_CK_3dhom_prefer)

print(f"idxCK_hom    = {idxCK_hom}   γ = {misfitsCK_hom['gamma'].iloc[idxCK_hom]:.2e}")
print(f"idxCK_swsw   = {idxCK_swsw}   γ = {misfitsCK_swsw['gamma'].iloc[idxCK_swsw]:.2e}")
print(f"idxCK_swhom  = {idxCK_swhom}   γ = {misfitsCK_swhom['gamma'].iloc[idxCK_swhom]:.2e}")
print(f"idxCK_3d3d   = {idxCK_3d3d}   γ = {misfitsCK_3d3d['gamma'].iloc[idxCK_3d3d]:.2e}")
print(f"idxCK_3dhom  = {idxCK_3dhom}   γ = {misfitsCK_3dhom['gamma'].iloc[idxCK_3dhom]:.2e}")


In [ ]:
# print(gammas_CK_hom[1:-1])
print(misfitsCK_hom['gamma'].tolist())

In [ ]:
print(len(misfitsCK_hom))
print(misfitsCK_hom)

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 1, figsize=(4,4), dpi=300)

ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=8)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=8)
ax.set_title("Synthetic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

i_start, i_end = 1, -1
i_start, i_end = 0, -2

ax.plot(misfitsCK_hom['m_mis'][i_start:i_end], misfitsCK_hom['d_mis'][i_start:i_end], 
        'o-', color='k', markerfacecolor=color_L[2],
        linewidth=1.0, markersize=4, label=r"HOM; $\rho$={:.1e}".format(rho_s))
for _, row in misfitsCK_hom.iloc[i_start:i_end].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='k', ha='left', va='bottom')

# ax.plot(misfitsCK_hom1['m_mis'][i_start:i_end], misfitsCK_hom1['d_mis'][i_start:i_end], 
#         'o-', color='b', markerfacecolor=color_L[2],
#         linewidth=1.0, markersize=4, label=r"HOM; $\rho$={:.1e}".format(rho_s1))
# for i, gamma in enumerate(misfitsCK_hom1['gamma'].iloc[i_start:i_end]):
#     ax.text(misfitsCK_hom1['m_mis'][i_start:i_end].iloc[i]+0.25, 
#             misfitsCK_hom1['d_mis'][i_start:i_end].iloc[i], 
#             f"{gamma:.0f}", fontsize=6, color='b')

# ax.text
# ax.plot(misfitsCK_swsw['m_mis'], misfitsCK_swsw['d_mis'], 's-', color='red', markerfacecolor=color_L[2],
#         linewidth=1.0, markersize=4, label=r"SW; SW")
# ax.plot(misfitsCK_swhom['m_mis'], misfitsCK_swhom['d_mis'], 's--', color='red', markerfacecolor=color_L[0],
#         linewidth=1.0, markersize=4, label=r"SW; HOM;")
# ax.plot(misfitsCK_3d3d['m_mis'], misfitsCK_3d3d['d_mis'], 'D-', color='dodgerblue', markerfacecolor=color_L[2],
#         linewidth=1.0, markersize=4, label=r"3D; 3D")
# ax.plot(misfitsCK_3dhom['m_mis'], misfitsCK_3dhom['d_mis'], 'D--', color='dodgerblue', markerfacecolor=color_L[0],
#         linewidth=1.0, markersize=4, label=r"3D; HOM;")

ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=8)

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 3, figsize=(12,4), dpi=300)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

color_L = ['silver', 'firebrick', 'white']


### (a), forward and inverse modeling in the same \mu model
ax = axes[0]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

print(f"idxCK_hom  = {idxCK_hom},  γ = {misfitsCK_hom['gamma'].iloc[idxCK_hom]:.2e}")

print(f"idxCK_swsw  = {idxCK_swsw},  γ = {misfitsCK_swsw['gamma'].iloc[idxCK_swsw]:.2e}")

print(f"idxCK_3d3d  = {idxCK_3d3d},  γ = {misfitsCK_3d3d['gamma'].iloc[idxCK_3d3d]:.2e}")

ax.plot(misfitsCK_hom['m_mis'][i_start:i_end], misfitsCK_hom['d_mis'][i_start:i_end], 
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H_H")
# idxCK_hom = 5
ax.plot(misfitsCK_hom.iloc[idxCK_hom]['m_mis'], misfitsCK_hom.iloc[idxCK_hom]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H_H preferred $\gamma$={:.1e}".format(misfitsCK_hom['gamma'].iloc[idxCK_hom]))
for _, row in misfitsCK_hom.iloc[i_start:i_end].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='k', ha='left', va='bottom')
    
ax.plot(misfitsCK_swsw['m_mis'][i_start:i_end], misfitsCK_swsw['d_mis'][i_start:i_end], 
    's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
# idxCK_swsw = 5
ax.plot(misfitsCK_swsw.iloc[idxCK_swsw]['m_mis'], misfitsCK_swsw.iloc[idxCK_swsw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S_S preferred $\gamma$={:.1e}".format(misfitsCK_swsw['gamma'].iloc[idxCK_swsw]))

ax.plot(misfitsCK_3d3d['m_mis'][i_start:i_end], misfitsCK_3d3d['d_mis'][i_start:i_end], 
    'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
# idxCK_3d3d = 5
ax.plot(misfitsCK_3d3d.iloc[idxCK_3d3d]['m_mis'], misfitsCK_3d3d.iloc[idxCK_3d3d]['d_mis'], 'D', color='dodgerblue', 
        markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_3D preferred $\gamma$={:.1e}".format(misfitsCK_3d3d['gamma'].iloc[idxCK_3d3d]))

ax.text(0.02, 0.02, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
# ax.grid(True, which='major', color='gray', linestyle='-', alpha=0.6 )
# ax.grid(True, which='minor', color='lightgray', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 75)
# ax.set_xlim(0.02, 0.09)


### (b), forward in the SW model, and inverse in either SW or HOM model 
ax = axes[1]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

print(f"idxCK_swsw  = {idxCK_swsw},  γ = {misfitsCK_swsw['gamma'].iloc[idxCK_swsw]:.2e}")

print(f"idxCK_swhom  = {idxCK_swhom},  γ = {misfitsCK_swhom['gamma'].iloc[idxCK_swhom]:.2e}")

ax.plot(misfitsCK_swsw['m_mis'][i_start:i_end], misfitsCK_swsw['d_mis'][i_start:i_end], 
    's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
# idxCK_swsw = 6
ax.plot(misfitsCK_swsw.iloc[idxCK_swsw]['m_mis'], misfitsCK_swsw.iloc[idxCK_swsw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S_S preferred $\gamma$={:.1e}".format(misfitsCK_swsw['gamma'].iloc[idxCK_swsw]))
for _, row in misfitsCK_swsw.iloc[i_start:i_end].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='red', ha='left', va='bottom')
    
ax.plot(misfitsCK_swhom['m_mis'][i_start:i_end], misfitsCK_swhom['d_mis'][i_start:i_end], 
    's--', color='red', markerfacecolor='lightgray', linewidth=1.0, markersize=4, label=r"S_H")
# idxCK_swhom = 6
ax.plot(misfitsCK_swhom.iloc[idxCK_swhom]['m_mis'], misfitsCK_swhom.iloc[idxCK_swhom]['d_mis'], 
        's', color='k', markerfacecolor='red', markersize=5, 
        label=r"S_H preferred $\gamma$={:.1e}".format(misfitsCK_swhom['gamma'].iloc[idxCK_swhom]))

ax.text(0.02, 0.02, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


### (c), forward in the SW model, and inverse in either SW or HOM model 
ax = axes[2]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

print(f"idxCK_3d3d  = {idxCK_3d3d},  γ = {misfitsCK_3d3d['gamma'].iloc[idxCK_3d3d]:.2e}")

print(f"idxCK_3dhom  = {idxCK_3dhom},  γ = {misfitsCK_3dhom['gamma'].iloc[idxCK_3dhom]:.2e}")

ax.plot(misfitsCK_3d3d['m_mis'][i_start:i_end], misfitsCK_3d3d['d_mis'][i_start:i_end], 
    'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
# idxCK_3d3d = 6
ax.plot(misfitsCK_3d3d.iloc[idxCK_3d3d]['m_mis'], misfitsCK_3d3d.iloc[idxCK_3d3d]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_3D preferred $\gamma$={:.1e}".format(misfitsCK_3d3d['gamma'].iloc[idxCK_3d3d]))
for _, row in misfitsCK_3d3d.iloc[i_start:i_end].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='dodgerblue', ha='left', va='bottom')
    
ax.plot(misfitsCK_3dhom['m_mis'][i_start:i_end], misfitsCK_3dhom['d_mis'][i_start:i_end], 
        'D--', color='dodgerblue', markerfacecolor=color_L[0], linewidth=1.0, markersize=4, label=r"3D_H")
# idxCK_3dhom = 6
ax.plot(misfitsCK_3dhom.iloc[idxCK_3dhom]['m_mis'], misfitsCK_3dhom.iloc[idxCK_3dhom]['d_mis'], 
        'D', color='k', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_H preferred $\gamma$={:.1e}".format(misfitsCK_3dhom['gamma'].iloc[idxCK_3dhom]))

ax.text(0.02, 0.02, panel_labels[2], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)

figpath = datadir + 'figures/synth/'
# output_file = figpath + f'{lc_prefix}{rho_s:.0e}_{meshnameCK}_{slip_pattern}_{pattern_option}.pdf'
output_file = figpath + f'Lcurvesynlockazbd_{meshnameCK}_{slip_pattern}_{pattern_option}.pdf'
print(output_file)
plt.savefig(output_file, dpi=300, bbox_inches='tight')


## Interseismic inversion parameters

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "rst_locking/"

# Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
obs_disp_name = "data/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
                          'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])
# Decide the weights of the horizontal, vertical components
f_h, f_v = 1, 1
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
w_h, w_v = int(1/f_h), int(1/f_v)

# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email

# Mesh selection (uneven-top mesh; legacy even-top branch removed)
meshnameCK = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface

def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
mu_upper = mu_expression(mu_u)

# Model identifiers
homo_str      = f"_mul{round(mu_expression(mu_b))}u{round(mu_expression(mu_b))}"
sw_str        = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
het3d_str_ori = f"_DeShon3D_ref_{round(1.0)}_hull"
contrast_factor = 2.5  # adopted since 03/05/2026
het3d_str     = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"
het3d_str_4   = f"_DeShon3D_ref1D_{round(4.0)}_hull"

CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
if CG_mu_deg == 2:
    het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"
    het3d_str     = het3d_str     + f"_CG{CG_mu_deg}"
    het3d_str_4   = het3d_str_4   + f"_CG{CG_mu_deg}"

rho_s_in = 2.5e8   # ~15 km, close to the maximum resolution

# Helper: read an interseismic L-curve summary (4-column format: d_mis m_mis gamma rho_s)
# and sort rows by gamma ascending so plot ordering matches the actual saved L-curve
# regardless of the order in which gammas_s was iterated by the inversion script.
def _load_lc_lock(mu_str):
    fname = f"Lcurvelockboth_rs{rho_s_in:.0e}_{meshnameCK}_{mu_str}.txt"
    print(fname)
    df = pd.read_csv(datadir + resultpath + fname, sep=r'\s+',
                     names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    return df.sort_values('gamma').reset_index(drop=True)

# Helper: row index whose gamma is closest to a requested preferred value
def _idx_for_gamma_lock(df, gamma_pref):
    return int((df['gamma'] - gamma_pref).abs().idxmin())

misfitsCK_hom_lock    = _load_lc_lock(homo_str)
misfitsCK_sw_lock     = _load_lc_lock(sw_str)
misfitsCK_3d_ori_lock = _load_lc_lock(het3d_str_ori)
misfitsCK_3d_lock     = _load_lc_lock(het3d_str)
misfitsCK_3d4_lock    = _load_lc_lock(het3d_str_4)

# Preferred gamma per model (closest-row picked from file)
gamma_CK_hom_prefer_lock    = 3e3
gamma_CK_sw_prefer_lock     = 3e3
gamma_CK_3dori_prefer_lock = 3e3
gamma_CK_3d_prefer_lock     = 3e3
gamma_CK_3d4_prefer_lock    = 3e3

idxCK_hom_lock    = _idx_for_gamma_lock(misfitsCK_hom_lock,    gamma_CK_hom_prefer_lock)
idxCK_sw_lock     = _idx_for_gamma_lock(misfitsCK_sw_lock,     gamma_CK_sw_prefer_lock)
idxCK_3d_ori_lock = _idx_for_gamma_lock(misfitsCK_3d_ori_lock, gamma_CK_3dori_prefer_lock)
idxCK_3d_lock     = _idx_for_gamma_lock(misfitsCK_3d_lock,     gamma_CK_3d_prefer_lock)
idxCK_3d4_lock    = _idx_for_gamma_lock(misfitsCK_3d4_lock,    gamma_CK_3d4_prefer_lock)

print(idxCK_hom_lock,    misfitsCK_hom_lock['gamma'].iloc[idxCK_hom_lock])
print(idxCK_sw_lock,     misfitsCK_sw_lock['gamma'].iloc[idxCK_sw_lock])
print(idxCK_3d_ori_lock, misfitsCK_3d_ori_lock['gamma'].iloc[idxCK_3d_ori_lock])
print(idxCK_3d_lock,     misfitsCK_3d_lock['gamma'].iloc[idxCK_3d_lock])
print(idxCK_3d4_lock,    misfitsCK_3d4_lock['gamma'].iloc[idxCK_3d4_lock])


In [ ]:
print(misfitsCK_hom_lock)

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 2, figsize=(8,4), dpi=300)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]


### (a), synthetics, forward and inverse modeling in the same \mu model
ax = axes[0]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)


ax.plot(misfitsCK_hom['m_mis'][i_start:i_end], misfitsCK_hom['d_mis'][i_start:i_end], 
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H_H")
# idxCK_hom = 5
ax.plot(misfitsCK_hom.iloc[idxCK_hom]['m_mis'], misfitsCK_hom.iloc[idxCK_hom]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H_H preferred $\gamma$={:.1e}".format(misfitsCK_hom['gamma'].iloc[idxCK_hom]))
for _, row in misfitsCK_hom.iloc[i_start:i_end].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='k', ha='left', va='bottom')

ax.plot(misfitsCK_swsw['m_mis'][i_start:i_end], misfitsCK_swsw['d_mis'][i_start:i_end], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
# idxCK_swsw = 5
ax.plot(misfitsCK_swsw.iloc[idxCK_swsw]['m_mis'], misfitsCK_swsw.iloc[idxCK_swsw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S_S preferred $\gamma$={:.1e}".format(misfitsCK_swsw['gamma'].iloc[idxCK_swsw]))

ax.plot(misfitsCK_3d3d['m_mis'][i_start:i_end], misfitsCK_3d3d['d_mis'][i_start:i_end], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
# idxCK_3d3d = 5
ax.plot(misfitsCK_3d3d.iloc[idxCK_3d3d]['m_mis'], misfitsCK_3d3d.iloc[idxCK_3d3d]['d_mis'], 'D', color='dodgerblue', 
        markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_3D preferred $\gamma$={:.1e}".format(misfitsCK_3d3d['gamma'].iloc[idxCK_3d3d]))

ax.text(0.02, 0.02, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
# ax.grid(True, which='major', color='gray', linestyle='-', alpha=0.6 )
# ax.grid(True, which='minor', color='lightgray', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 75)
# ax.set_xlim(0.02, 0.09)


### (b), real interseismic, forward and inverse modeling in the same \mu model
ax = axes[1]
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Real interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

i_start_i, i_end_i = 0, None

ax.plot(misfitsCK_hom_lock['m_mis'][i_start_i:i_end_i], misfitsCK_hom_lock['d_mis'][i_start_i:i_end_i], 
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H")
# idxCK_hom = 6
ax.plot(misfitsCK_hom_lock.iloc[idxCK_hom_lock]['m_mis'], misfitsCK_hom_lock.iloc[idxCK_hom_lock]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H preferred $\gamma$={:.1e}".format(misfitsCK_hom_lock['gamma'].iloc[idxCK_hom_lock]))
for _, row in misfitsCK_hom_lock.iloc[i_start_i:i_end_i].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='k', ha='left', va='bottom')
    
ax.plot(misfitsCK_sw_lock['m_mis'][i_start_i:i_end_i], misfitsCK_sw_lock['d_mis'][i_start_i:i_end_i], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S")
# idxCK_sw = 6
ax.plot(misfitsCK_sw_lock.iloc[idxCK_sw_lock]['m_mis'], misfitsCK_sw_lock.iloc[idxCK_sw_lock]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S preferred $\gamma$={:.1e}".format(misfitsCK_sw_lock['gamma'].iloc[idxCK_sw_lock]))

ax.plot(misfitsCK_3d_lock['m_mis'][i_start_i:i_end_i], misfitsCK_3d_lock['d_mis'][i_start_i:i_end_i], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D")
# idxCK_3d = 6
ax.plot(misfitsCK_3d_lock.iloc[idxCK_3d_lock]['m_mis'], misfitsCK_3d_lock.iloc[idxCK_3d_lock]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_lock['gamma'].iloc[idxCK_3d_lock]))

ax.text(0.02, 0.02, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

ax.plot(misfitsCK_3d_ori_lock['m_mis'][i_start_i:i_end_i], misfitsCK_3d_ori_lock['d_mis'][i_start_i:i_end_i], 
        'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Orig. 3D")
# idxCK_3d_ori = 6
ax.plot(misfitsCK_3d_ori_lock.iloc[idxCK_3d_ori_lock]['m_mis'], misfitsCK_3d_ori_lock.iloc[idxCK_3d_ori_lock]['d_mis'], 
        'D', color='darkblue', markerfacecolor='darkblue', markersize=5, 
        label=r"Orig. 3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_ori_lock['gamma'].iloc[idxCK_3d_ori_lock]))


# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 80)
# ax.set_xlim(0, 0.1)

figpath = datadir + 'figures/synth/'
# output_file = figpath + f'{lc_prefix}{rho_s:.0e}_and_lock_{meshnameCK}_{slip_pattern}_{pattern_option}.pdf'
output_file = figpath + f'Lcurvesynlockazbd_and_lock_{meshnameCK}_{slip_pattern}_{pattern_option}.pdf'
print(output_file)
plt.savefig(output_file, dpi=300, bbox_inches='tight')


## Az-constrained interseismic L-curves


In [ ]:
# Az-constrained interseismic L-curves
# Files: Lcurvelockbothaz{az}{amp_tag}_rs{rho:.0e}_{mesh}{mu_str}.txt
# amp_tag: ''  -> bounded tanh   m_phys = amp_max*(tanh(m)+1)/2 ∈ [0, amp_max]
# amp_tag: 'u' -> unbounded linear m_phys = m                    (no bound)
# azimuth_az = 33.5
azimuth_az = 45
az_str     = f"{azimuth_az:.0f}"

het3d_str_ori = f"_DeShon3D_ref_{round(1.0)}_hull"
if CG_mu_deg == 2:
    het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"

# Flip this to load the unbounded (use_tanh_amp=False) runs instead of the bounded ones.
use_tanh_amp = False
amp_tag      = '' if use_tanh_amp else 'u'

rho_s_az   = 2.5e8

def _load_lc_lock_az(mu_str):
    fname = f"Lcurvelockbothaz{az_str}{amp_tag}_rs{rho_s_az:.0e}_{meshnameCK}{mu_str}.txt"
    full = datadir + resultpath + fname
    if not os.path.exists(full):
        print(f"[missing] {fname}")
        return None
    print(fname)
    df = pd.read_csv(full, sep=r'\s+',
                     names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    df = df.drop_duplicates(subset='gamma')
    return df.sort_values('gamma').reset_index(drop=True)

misfitsCK_hom_az = _load_lc_lock_az(homo_str)
misfitsCK_sw_az  = _load_lc_lock_az(sw_str)
misfitsCK_3d_az  = _load_lc_lock_az(het3d_str)
misfitsCK_3d_ori_az  = _load_lc_lock_az(het3d_str_ori)

print(f"amp mode: {'tanh-bounded' if use_tanh_amp else 'linear-unbounded'}  (amp_tag={amp_tag!r})")
if misfitsCK_hom_az is not None: print("Hom K_2LAYER γ:", misfitsCK_hom_az['gamma'].tolist())
if misfitsCK_sw_az  is not None: print("Het K_2LAYER γ:", misfitsCK_sw_az['gamma'].tolist())
if misfitsCK_3d_az  is not None: print("3D Het DeShon γ:", misfitsCK_3d_az['gamma'].tolist())
if misfitsCK_3d_ori_az  is not None: print("3D Het DeShon (original) γ:", misfitsCK_3d_ori_az['gamma'].tolist())


# Preferred gamma per model (closest-row picked from file)
gamma_CK_hom_prefer_az    = 4e3
gamma_CK_sw_prefer_az     = 4e3
gamma_CK_3d_ori_prefer_az = 4e3
gamma_CK_3d_prefer_az     = 4e3

idxCK_hom_az    = _idx_for_gamma_lock(misfitsCK_hom_az,    gamma_CK_hom_prefer_az)
idxCK_sw_az     = _idx_for_gamma_lock(misfitsCK_sw_az,     gamma_CK_sw_prefer_az)
idxCK_3d_ori_az = _idx_for_gamma_lock(misfitsCK_3d_ori_az, gamma_CK_3d_ori_prefer_az)
idxCK_3d_az     = _idx_for_gamma_lock(misfitsCK_3d_az,     gamma_CK_3d_prefer_az)


In [ ]:
# Plot Az-constrained interseismic L-curves
fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)

color_L = ['silver', 'firebrick', 'white']
i_start_az, i_end_az = 1, None

ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
title_mode = "tanh-bounded" if use_tanh_amp else "linear-unbounded"
ax.set_title(f"Azimuth-constrained real interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

if misfitsCK_hom_az is not None:
    ax.plot(misfitsCK_hom_az['m_mis'][i_start_az:i_end_az],
            misfitsCK_hom_az['d_mis'][i_start_az:i_end_az],
            'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4,
            label=r"H")
    ax.plot(misfitsCK_hom_az.iloc[idxCK_hom_az]['m_mis'], misfitsCK_hom_az.iloc[idxCK_hom_az]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H preferred $\gamma$={:.1e}".format(misfitsCK_hom_az['gamma'].iloc[idxCK_hom_az]))

    for _, row in misfitsCK_hom_az.iloc[i_start_az:i_end_az].iterrows():
        ax.annotate(f"{row['gamma']:.0f}",
                    xy=(row['m_mis'], row['d_mis']),
                    xytext=(2, 2), textcoords='offset points',
                    fontsize=6, color='k')

if misfitsCK_sw_az is not None:
    ax.plot(misfitsCK_sw_az['m_mis'][i_start_az:i_end_az],
            misfitsCK_sw_az['d_mis'][i_start_az:i_end_az],
            's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4,
            label=r"S")
    ax.plot(misfitsCK_sw_az.iloc[idxCK_sw_az]['m_mis'], misfitsCK_sw_az.iloc[idxCK_sw_az]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S preferred $\gamma$={:.1e}".format(misfitsCK_sw_az['gamma'].iloc[idxCK_sw_az]))
    # for _, row in misfitsCK_sw_az.iloc[i_start_az:i_end_az].iterrows():
    #     ax.annotate(f"{row['gamma']:.0f}",
    #                 xy=(row['m_mis'], row['d_mis']),
    #                 xytext=(2, 2), textcoords='offset points',
    #                 fontsize=6, color='red')

if misfitsCK_3d_az is not None:
    ax.plot(misfitsCK_3d_az['m_mis'][i_start_az:i_end_az],
            misfitsCK_3d_az['d_mis'][i_start_az:i_end_az],
            'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4,
            label=r"3D")
    ax.plot(misfitsCK_3d_az.iloc[idxCK_3d_az]['m_mis'], misfitsCK_3d_az.iloc[idxCK_3d_az]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_az['gamma'].iloc[idxCK_3d_az]))
    # for _, row in misfitsCK_3d_az.iloc[i_start_az:i_end_az].iterrows():
    #     ax.annotate(f"{row['gamma']:.0f}",
    #                 xy=(row['m_mis'], row['d_mis']),
    #                 xytext=(2, 2), textcoords='offset points',
    #                 fontsize=6, color='dodgerblue')

if misfitsCK_3d_ori_az is not None:
        ax.plot(misfitsCK_3d_ori_az['m_mis'][i_start_az:i_end_az],
                misfitsCK_3d_ori_az['d_mis'][i_start_az:i_end_az],
                'D-', color='darkblue', markerfacecolor=color_L[0], linewidth=1.0, markersize=4,
                label=r"Orig. 3D")
        ax.plot(misfitsCK_3d_ori_az.iloc[idxCK_3d_ori_az]['m_mis'], misfitsCK_3d_ori_az.iloc[idxCK_3d_ori_az]['d_mis'], 
                'D', color='darkblue', markerfacecolor='darkblue', markersize=5, 
                label=r"Orig. 3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_ori_az['gamma'].iloc[idxCK_3d_ori_az]))
        # for _, row in misfitsCK_3d_ori_az.iloc[i_start_az:i_end_az].iterrows():
        #     ax.annotate(f"{row['gamma']:.0f}",
        #                 xy=(row['m_mis'], row['d_mis']),
        #                 xytext=(3, 3), textcoords='offset points',
        #                 fontsize=6, color='darkblue')

ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6)
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2)
ax.set_box_aspect(1)
ax.legend(fontsize=7)

figpath = datadir + 'figures/synth/'
# output_file = figpath + f'Lcurvelockbothaz{az_str}{amp_tag}_rs{rho_s_az:.0e}_{meshnameCK}.pdf'
output_file = figpath + f'Lcurvelockbothaz{az_str}{amp_tag}_{meshnameCK}.pdf'
print(output_file)
plt.savefig(output_file, dpi=300, bbox_inches='tight')


## Coseismic inversion parameters

In [ ]:
# Define the inversion results path
resultpath_co = "rst_coseis/"

# coseismic data file
obs_disp_name_co = "data/Protti_etal_2014_tableS1.csv"

df_co = pd.read_csv(datadir + obs_disp_name_co, sep=",", skiprows=1, \
                 names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                        'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])

rho_s_co = 2e7   # ~4.5 km, close to the maximum resolution

# Same helper pattern as the interseismic block, just with the coseismic prefix / rho.
def _load_lc_co(mu_str):
    fname = f"Lcurvecoseis_rs{rho_s_co:.0e}_{meshnameCK}_{mu_str}.txt"
    print(fname)
    df = pd.read_csv(datadir + resultpath_co + fname, sep=r'\s+',
                     names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    return df.sort_values('gamma').reset_index(drop=True)

def _idx_for_gamma_co(df, gamma_pref):
    return int((df['gamma'] - gamma_pref).abs().idxmin())

misfitsCK_hom_co    = _load_lc_co(homo_str)
misfitsCK_sw_co     = _load_lc_co(sw_str)
misfitsCK_3d_ori_co = _load_lc_co(het3d_str_ori)
misfitsCK_3d_co     = _load_lc_co(het3d_str)
# misfitsCK_3d4_co  = _load_lc_co(het3d_str_4)   # uncomment when the x4 L-curve file exists

#### Preferred gammas
## below roughly point to the corners
# gamma_CK_hom_prefer_co    = 6e2
# gamma_CK_sw_prefer_co     = 6e2
# gamma_CK_3d_ori_prefer_co = 6e2
# gamma_CK_3d_prefer_co     = 6e2

## below to match similar potency to KN16
gamma_CK_hom_prefer_co    = 2.5e2
gamma_CK_3d_ori_prefer_co = 2.5e2
gamma_CK_sw_prefer_co     = 3e2
gamma_CK_3d_prefer_co     = 3e2

idxCK_hom_co    = _idx_for_gamma_co(misfitsCK_hom_co,    gamma_CK_hom_prefer_co)
idxCK_sw_co     = _idx_for_gamma_co(misfitsCK_sw_co,     gamma_CK_sw_prefer_co)
idxCK_3d_ori_co = _idx_for_gamma_co(misfitsCK_3d_ori_co, gamma_CK_3d_ori_prefer_co)
idxCK_3d_co     = _idx_for_gamma_co(misfitsCK_3d_co,     gamma_CK_3d_prefer_co)

print(idxCK_hom_co,    misfitsCK_hom_co['gamma'].iloc[idxCK_hom_co])
print(idxCK_sw_co,     misfitsCK_sw_co['gamma'].iloc[idxCK_sw_co])
print(idxCK_3d_ori_co, misfitsCK_3d_ori_co['gamma'].iloc[idxCK_3d_ori_co])
print(idxCK_3d_co,     misfitsCK_3d_co['gamma'].iloc[idxCK_3d_co])


In [ ]:
print(misfitsCK_hom_co)

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 3, figsize=(12,4), dpi=300)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

color_L = ['silver', 'firebrick', 'white']


### (a), forward in the SW model, and inverse in either SW or HOM model 
ax = axes[0]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

ax.plot(misfitsCK_swsw['m_mis'][i_start:i_end], misfitsCK_swsw['d_mis'][i_start:i_end], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S_S")
# idxCK_swsw = 6
ax.plot(misfitsCK_swsw.iloc[idxCK_swsw]['m_mis'], misfitsCK_swsw.iloc[idxCK_swsw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S_S preferred $\gamma$={:.1e}".format(misfitsCK_swsw['gamma'].iloc[idxCK_swsw]))
for _, row in misfitsCK_swsw.iloc[i_start:i_end].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='red', ha='left', va='bottom')
    
ax.plot(misfitsCK_swhom['m_mis'][i_start:i_end], misfitsCK_swhom['d_mis'][i_start:i_end], 
        's--', color='red', markerfacecolor='lightgray', linewidth=1.0, markersize=4, label=r"S_H")
# idxCK_swhom = 6
ax.plot(misfitsCK_swhom.iloc[idxCK_swhom]['m_mis'], misfitsCK_swhom.iloc[idxCK_swhom]['d_mis'], 
        's', color='k', markerfacecolor='red', markersize=5, 
        label=r"S_H preferred $\gamma$={:.1e}".format(misfitsCK_swhom['gamma'].iloc[idxCK_swhom]))

ax.text(0.02, 0.02, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


### (b), forward in the SW model, and inverse in either SW or HOM model 
ax = axes[1]
# ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Synthetic interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

ax.plot(misfitsCK_3d3d['m_mis'][i_start:i_end], misfitsCK_3d3d['d_mis'][i_start:i_end], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D_3D")
# idxCK_3d3d = 6
ax.plot(misfitsCK_3d3d.iloc[idxCK_3d3d]['m_mis'], misfitsCK_3d3d.iloc[idxCK_3d3d]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_3D preferred $\gamma$={:.1e}".format(misfitsCK_3d3d['gamma'].iloc[idxCK_3d3d]))
for _, row in misfitsCK_3d3d.iloc[i_start:i_end].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='dodgerblue', ha='left', va='bottom')
    
ax.plot(misfitsCK_3dhom['m_mis'][i_start:i_end], misfitsCK_3dhom['d_mis'][i_start:i_end], 
        'D--', color='dodgerblue', markerfacecolor=color_L[0], linewidth=1.0, markersize=4, label=r"3D_H")
# idxCK_3dhom = 6
ax.plot(misfitsCK_3dhom.iloc[idxCK_3dhom]['m_mis'], misfitsCK_3dhom.iloc[idxCK_3dhom]['d_mis'], 
        'D', color='k', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D_H preferred $\gamma$={:.1e}".format(misfitsCK_3dhom['gamma'].iloc[idxCK_3dhom]))

ax.text(0.02, 0.02, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


### (c), real coseismic inversions
ax = axes[2]
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
# ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Real coseismic slip inversion", fontsize=9)
ax.tick_params(labelsize=8)

i_start_c, i_end_c = 0, None

color_L = ['silver', 'firebrick', 'white']

ax.plot(misfitsCK_hom_co['m_mis'][i_start_c:i_end_c], misfitsCK_hom_co['d_mis'][i_start_c:i_end_c], 
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H")
# idxCK_hom = 6
ax.plot(misfitsCK_hom_co.iloc[idxCK_hom_co]['m_mis'], misfitsCK_hom_co.iloc[idxCK_hom_co]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"H preferred $\gamma$={:.1e}".format(misfitsCK_hom_co['gamma'].iloc[idxCK_hom_co]))
for _, row in misfitsCK_hom_co.iloc[i_start_c:i_end_c].iterrows():
    ax.annotate(f"{row['gamma']:.0f}",
                xy=(row['m_mis'], row['d_mis']),
                xytext=(2, 2), textcoords='offset points',
                fontsize=6, color='k', ha='left', va='bottom')
    
ax.plot(misfitsCK_sw_co['m_mis'][i_start_c:i_end_c], misfitsCK_sw_co['d_mis'][i_start_c:i_end_c], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S")
# idxCK_sw = 6
ax.plot(misfitsCK_sw_co.iloc[idxCK_sw_co]['m_mis'], misfitsCK_sw_co.iloc[idxCK_sw_co]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"S preferred $\gamma$={:.1e}".format(misfitsCK_sw_co['gamma'].iloc[idxCK_sw_co]))

ax.plot(misfitsCK_3d_co['m_mis'][i_start_c:i_end_c], misfitsCK_3d_co['d_mis'][i_start_c:i_end_c], 
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D")
# idxCK_3d = 6
ax.plot(misfitsCK_3d_co.iloc[idxCK_3d_co]['m_mis'], misfitsCK_3d_co.iloc[idxCK_3d_co]['d_mis'], 
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5, 
        label=r"3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_co['gamma'].iloc[idxCK_3d_co]))

ax.plot(misfitsCK_3d_ori_co['m_mis'][i_start_c:i_end_c], misfitsCK_3d_ori_co['d_mis'][i_start_c:i_end_c], 
        'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Orig. 3D")
# idxCK_3d_ori = 6
ax.plot(misfitsCK_3d_ori_co.iloc[idxCK_3d_ori_co]['m_mis'], misfitsCK_3d_ori_co.iloc[idxCK_3d_ori_co]['d_mis'], 
        'D', color='darkblue', markerfacecolor='darkblue', markersize=5, 
        label=r"Orig. 3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_ori_co['gamma'].iloc[idxCK_3d_ori_co]))

ax.text(0.02, 0.02, panel_labels[2], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')

# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 80)
# ax.set_xlim(0, 0.1)

figpath = datadir + 'figures/synth/'
# output_file = figpath + f'{lc_prefix}{rho_s:.0e}_and_coseis_{meshnameCK}_{slip_pattern}_{pattern_option}.pdf'
output_file = figpath + f'Lcurvesynlockazbd_and_coseis_{meshnameCK}_{slip_pattern}_{pattern_option}.pdf'
print(output_file)
plt.savefig(output_file, dpi=300, bbox_inches='tight')
